# 01 · Carga y calidad del reporte del Miner

**Objetivo.** Cargar un reporte del Miner de forma tolerante, describir la
organización y el origen, medir la cobertura y auditar la calidad/coherencia de
los datos antes de analizarlos.

**Entrada.** `INPUT_PATHS`, lista de reportes que se fusionan por repositorio.
Por defecto: `results.json` (scan) si existe; si no, todos los que existan de
`results-sbom.json` y `results-vuln.json`, en ese orden.

**Salida.** Ninguna escritura a disco: notebook de exploración y validación.

**Supuestos.**
- El reporte puede venir de `scan`, `sbom` o `vuln`; algunas secciones pueden
  faltar y no se asume su presencia.
- Todos los cálculos vienen de `analysis.loader` y `analysis.metrics`; el
  notebook no duplica lógica.

In [ ]:
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    return start


def _default_inputs(root: Path) -> list:
    scan = root / "results.json"
    if scan.is_file():
        return [scan]
    candidates = [root / "results-sbom.json", root / "results-vuln.json"]
    return [path for path in candidates if path.is_file()]


REPO_ROOT = _find_repo_root(Path.cwd())
INPUT_PATHS = _default_inputs(REPO_ROOT)
OUTPUT_DIR = REPO_ROOT / "analysis" / "outputs"

if not INPUT_PATHS:
    raise FileNotFoundError(
        "No se encontró results.json, results-sbom.json ni results-vuln.json"
    )

print("REPO_ROOT :", REPO_ROOT)
print("INPUT_PATHS:")
for path in INPUT_PATHS:
    print("  -", path)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
import json
import sys

import matplotlib

matplotlib.use("Agg")  # backend no interactivo (sin display)
import pandas as pd

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis import metrics
from analysis.loader import (
    SBOM_OK_STATUSES,
    VULN_OK_STATUSES,
    load_report,
    merge_reports,
    to_records,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

## 1. Carga del reporte

`load_report` normaliza el JSON de forma tolerante: nunca asume que existan
`findings`, `vulnerabilities` ni `sbom`, y acumula advertencias en
`report.warnings`.

In [ ]:
reports = [load_report(path) for path in INPUT_PATHS]
report = merge_reports(reports)

print("Reportes de entrada:")
for path, partial in zip(INPUT_PATHS, reports):
    print(
        f"  - {path.name}: source_kind={partial.source_kind}, "
        f"repos={len(partial.repositories)}"
    )
print()
print("Organización :", report.organization)
print("Origen       :", report.source_kind, f"({report.source_path})")
print("Repositorios :", len(report.repositories))
print("Advertencias :", len(report.warnings))
print()
print("summary fusionado (incluye 'components'):")
print(json.dumps(report.summary, indent=2, ensure_ascii=False))

## 2. Cobertura y estados

`compute_coverage` responde: *¿qué proporción de la organización tiene al menos
un análisis exitoso y qué modos de fallo aparecen?* Un repositorio cuenta como
cubierto si `status == "analyzed"` o si su SBOM/Grype terminó con éxito.

In [ ]:
coverage = metrics.compute_coverage(report)

print("repositories_total:", coverage["repositories_total"])
print("coverage_ratio    :", coverage["coverage_ratio"])
print("unsupported       :", coverage["unsupported"])
print("vuln_failed       :", coverage["vuln_failed"])
print("sbom_failed       :", coverage["sbom_failed"])

for label, key in [
    ("Estado de repositorio", "by_repo_status"),
    ("Estado de Grype", "by_vuln_status"),
    ("Estado de SBOM", "by_sbom_status"),
]:
    dist = pd.DataFrame(
        {"estado": list(coverage[key]), "repos": list(coverage[key].values())}
    )
    print(f"\n{label}:")
    print(dist.to_string(index=False) if not dist.empty else "  (sin datos)")

## 3. Repositorios y distribución por severidad

Tabla plana de repositorios (`to_records`) y distribución global de severidades
(las seis severidades canónicas de Grype, incluidas las que valen cero).

In [ ]:
records = to_records(report)
repos_df = pd.DataFrame(records["repositories"])

print(f"Repositorios: {len(repos_df)} filas")
print(repos_df.to_string(index=False) if not repos_df.empty else "(sin repositorios)")

In [ ]:
sev_df = pd.DataFrame(metrics.compute_severity_distribution(records))
total_vulns = int(sev_df["count"].sum())
sev_df["percent"] = (
    (sev_df["count"] / total_vulns * 100).round(2) if total_vulns else 0.0
)

print(f"Distribución por severidad (total={total_vulns}):")
print(sev_df.to_string(index=False))

## 4. Chequeos de calidad

Se revisan campos nulos por tabla y la coherencia entre los totales del
`summary`, `vuln_total`, las filas de detalle y la distribución de severidad.
Una discrepancia **no se corrige**: se reporta.

In [ ]:
vuln_df = pd.DataFrame(records["vulnerabilities"])
find_df = pd.DataFrame(records["findings"])


def _nulls(df: pd.DataFrame) -> pd.Series:
    if df.empty:
        return pd.Series({col: 0 for col in df.columns}, dtype=int)
    return df.isna().sum().astype(int)


for name, df in [
    ("repositories", repos_df),
    ("findings", find_df),
    ("vulnerabilities", vuln_df),
]:
    if df.empty:
        print(f"{name}: sin filas ({len(df)})")
        continue
    counts = _nulls(df)
    pending = counts[counts > 0]
    print(f"Nulos en {name} ({len(df)} filas):")
    print(pending.to_string() if not pending.empty else "  (ninguno)")

In [ ]:
summary = report.summary
flat_vulns = len(records["vulnerabilities"])
sev_sum = (
    int(pd.DataFrame(records["severity_distribution"])["count"].sum())
    if records["severity_distribution"]
    else 0
)
detail_total = sum(repo.vuln_total for repo in report.repositories)
components_detail = sum(repo.sbom_components for repo in report.repositories)
summary_vulns = summary.get("vulnerabilities")
summary_repos = summary.get("repositories")
summary_components = summary.get("components")
has_sbom = any(
    repo.sbom_components > 0 or repo.sbom_status in SBOM_OK_STATUSES
    for repo in report.repositories
)


def _check(condition) -> str:
    return "OK" if condition else "FALLA"


checks = [
    (
        "len(repositories) == coverage.repositories_total",
        len(report.repositories) == coverage["repositories_total"],
    ),
    (
        "sum(severity_distribution) == filas de vulnerabilidades",
        sev_sum == flat_vulns,
    ),
    ("sum(vuln_total) == filas de vulnerabilidades", detail_total == flat_vulns),
    (
        "summary.repositories == len(repositories)",
        summary_repos == len(report.repositories)
        if isinstance(summary_repos, int)
        else None,
    ),
    (
        "summary.vulnerabilities == filas de vulnerabilidades",
        summary_vulns == flat_vulns if isinstance(summary_vulns, int) else None,
    ),
    (
        "sum(sbom_components) == summary.components",
        components_detail == summary_components
        if (has_sbom and isinstance(summary_components, int))
        else None,
    ),
]

checks_df = pd.DataFrame(
    [
        {
            "chequeo": name,
            "resultado": "n/d" if ok is None else _check(ok),
        }
        for name, ok in checks
    ]
)
print(checks_df.to_string(index=False))

## 5. Resumen de calidad

In [ ]:
failures = int((checks_df["resultado"] == "FALLA").sum())
print(f"Repositorios          : {len(report.repositories)}")
print(f"Origen                : {report.source_kind}")
print(f"Cobertura de análisis : {coverage['coverage_ratio']:.1%}")
print(f"Vulnerabilidades      : {flat_vulns}")
print(f"Componentes SBOM      : {components_detail}")
print(f"Hallazgos CodeQL      : {len(records['findings'])}")
print(f"Chequeos fallidos     : {failures}")
print(f"Advertencias          : {len(coverage['warnings'])}")
if failures == 0:
    print("\nCalidad: sin incoherencias detectadas en los chequeos aplicables.")
else:
    print("\nCalidad: revisar los chequeos marcados como FALLA.")